# Summary
## TLDR; the DataFrame allocation is SLOW, so this is impractical
Tried using data frames recursively and they're terrible for performance.  An 11 person hierarchy with 4-5 levels took 6+ minutes to calculate.

Unacceptable, so trying with temporary tables or views...

In [0]:
from pyspark.sql import functions as fn, DataFrame, types as ty

In [0]:
df_staff = spark.createDataFrame([
        (1, "Al", "President", None),

        (2, "Betty", "Director", 1),
        (3, "Carl", "Manager", 2),
        (4, "Diane", "Engineer", 3),
        (5, "Evan", "Engineer", 3),

        (6, "Frank", "Director", 1),
        (7, "Gina", "Manager", 6),
        (8, "Harry", "Developer", 7),
        (9, "Irene", "Developer", 7),
        
        (10, "Jack", "Senior Developer", 7),
        (11, "Karen", "Developer", 10)
        ]
    , ["id", "name", "title", "reports_to"])

display(df_staff.select("*", fn.concat(fn.col("reports_to"), fn.lit("...")).alias("reports_2")))

Pass in the dataframe and NEVER change it
pull the top level (null relationship) rows as beginning of hierarchy
union to this the next level, as passed back from the recursive function

In [0]:
# Function to build a hierarchy given an source data frame and an child / parent pair of fields within it
def build_hierarchy(dfSource: DataFrame, child, parent):

    msg = ""

    # Add level and lineage columns if they don't exist
    if not dfSource.columns.__contains__("level"):
        dfSource = dfSource.withColumn("level", fn.lit(None))
    if not dfSource.columns.__contains__("lineage"):
        dfSource = dfSource.withColumn("lineage", fn.lit(None))

    # The starting hierarchy for this iteration is anything that already has a level defined
    hierarchy = dfSource.filter(
        f"level is not null"
        )
    
    # If hierarchy is empty, this is the first pass.  
    # Create new from rows with NULL in parent field
    if hierarchy.count() == 0:
        msg += "\nStarting hierarchy from beginning"
        hierarchy = dfSource.filter(
            f"{parent} is null"
            ).withColumns({
                "level": fn.lit(0),
                "lineage": fn.col("name")
            })

    while dfSource.filter("level is null").count() > 0:
        # Get currently highest level
        level = 1 + hierarchy.select(fn.max("level")).collect()[0][0]
        msg += f"\n\tBuilding level: {level}"

        # Add rows whose parents are at the currently deepest level of the hierarchy
        hierarchy = hierarchy.union(
            dfSource.alias("src").join(
                hierarchy.alias("h"),
                fn.col(f"src.{parent}") == fn.col(f"h.{child}"),
            "inner"
            ).select(
                "src.*"
            ).withColumns({
                "level": fn.lit(level),
                "lineage": fn.concat(fn.lit(" <- "), fn.col("src.name"))
            }
            )
        )
        
        # Combine the hierarchy with what remains
        dfSource = dfSource.alias("src").join(
                hierarchy.alias("x"),
                fn.col(f"src.{child}") == fn.col(f"x.{child}"),
                "leftanti"
        ).select(
            "src.*"
        ).union(hierarchy)

        # display(dfSource.withColumn("Intermediate", fn.lit("dfSource")))

        msg += f"\n\tRecursing to level {level + 1}"

    print(msg)
    return dfSource


In [0]:
# fin = build_hierarchy(build_hierarchy(df_staff, "id", "reports_to"), "id", "reports_to")
fin = build_hierarchy(df_staff, "id", "reports_to")
display(fin.sort("level", descending=True).sort("name"))